# Building an Agent Loop with Claude (Tool Use / Function Calling)

In this notebook, you will learn how to build an **Agent Loop** with Anthropic's Claude using the official Python SDK.

---

### What is an Agent Loop?
An agent loop is a multi-turn conversation cycle where Claude can pause generation, request external data or actions by calling a defined **tool**, receive the results, and continue reasoning until it reaches a final answer.

```text
  ┌─────────────────┐
  │   User Prompt   │
  └────────┬────────┘
           │
           ▼
  ┌────────────────────────────────────────────────────────┐
  │  Call Claude Messages API (with tools & conversation)  │◄────────┐
  └────────────────────────┬───────────────────────────────┘         │
                           │                                         │
                           ▼                                         │
                 [ Inspect stop_reason ]                             │
                           │                                         │
            ┌──────────────┴──────────────┐                          │
     "tool_use"                      "end_turn"                      │
            │                             │                          │
            ▼                             ▼                          │
  ┌────────────────────┐        ┌──────────────────┐                 │
  │ Execute local tool │        │  Print & return  │                 │
  │     (run_tool)     │        │   final answer   │                 │
  └─────────┬──────────┘        └──────────────────┘                 │
            │                                                        │
            ▼                                                        │
  ┌───────────────────────────────────────────────┐                  │
  │ Append tool_result to messages conversation   │──────────────────┘
  └───────────────────────────────────────────────┘
```

> ⚠️ **Credit Note:** Do not execute the request cells unless you want to spend API tokens.

## Step 1: Imports and Environment Setup

We use `load_dotenv(find_dotenv())` to automatically locate the `.env` file at the root of the project.

In [ ]:
import os
import anthropic
from dotenv import load_dotenv, find_dotenv

# Locate and load the root .env file
load_dotenv(find_dotenv())

api_key = os.getenv("ANTHROPIC_API_KEY") or os.getenv("API_KEY")
if not api_key:
    raise ValueError("API Key not found! Please check your .env file.")

# Instantiate the Anthropic client
client = anthropic.Anthropic(api_key=api_key)
print("✅ Anthropic client initialized successfully.")

## Step 2: Defining Available Tools with JSON Schema

Claude doesn't run your Python code directly. Instead, you declare the tools you offer through a **JSON Schema** definition.

Every tool definition requires three main fields:
1. **`name`**: Unique identifier for the tool (alphanumeric and underscores).
2. **`description`**: Detailed instructions on what the tool does and when to call it. Claude uses this description to decide whether tool invocation is needed.
3. **`input_schema`**: A standard JSON Schema defining properties, data types, and required parameters.

In [ ]:
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather and temperature for a given city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city, e.g. Austin, TX",
                }
            },
            "required": ["city"],
        },
    }
]

## Step 3: Implement the Tool Dispatcher (`run_tool`)

When Claude requests a tool, our Python code parses the arguments, executes the local Python function, and returns a string response.

In [ ]:
def run_tool(name: str, tool_input: dict) -> str:
    """Local dispatcher to execute the tool requested by Claude."""
    if name == "get_weather":
        city = tool_input.get("city", "Unknown")
        # In production, query a real weather API (e.g., OpenWeatherMap)
        return f"Weather in {city}: 95°F, Sunny with light winds."
    
    raise ValueError(f"Unknown tool requested: {name}")

## Step 4: The Agent Execution Loop

The core of any agent workflow is the evaluation of `response.stop_reason`:

- **`stop_reason == "tool_use"`**:
  Claude has identified that it needs data from one of your tools. Its response will contain one or more `ToolUseBlock` elements with `.name`, `.id`, and `.input`.
  
  **Required Return Format:**
  1. Append Claude's assistant turn containing the tool requests: `{"role": "assistant", "content": response.content}`.
  2. Append your tool execution results as a user turn:
     ```python
     {"role": "user", "content": [
         {"type": "tool_result", "tool_use_id": block.id, "content": result}
     ]}
     ```

- **`stop_reason == "end_turn"`**:
  Claude has finished processing and answered the question. Exit the loop.

- **Safety Guard (`max_iterations`)**:
  Always cap the number of iterations to prevent infinite loops and runaway credit consumption.

In [ ]:
# User prompt that requires external real-time data
messages = [
    {"role": "user", "content": "What should I wear in Austin today?"}
]

max_iterations = 5
iteration = 0

while iteration < max_iterations:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")

    # Call Claude with current message history and tool declarations
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )

    # Case 1: Claude is finished
    if response.stop_reason == "end_turn":
        print("\n[Final Answer from Claude]:")
        for block in response.content:
            if block.type == "text":
                print(block.text)
        break

    # Case 2: Claude requests tool execution
    elif response.stop_reason == "tool_use":
        tool_results = []

        # Execute each requested tool
        for block in response.content:
            if block.type == "text":
                print(f"Claude thought: {block.text}")
            elif block.type == "tool_use":
                print(f"Claude requests tool: {block.name} with arguments: {block.input}")
                
                result = run_tool(block.name, block.input)
                print(f"Tool returned: {result}")
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        # Update conversation history with assistant request + user tool_result
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    else:
        print(f"Loop terminated with stop reason: {response.stop_reason}")
        break

## Step 5: Key Takeaways for Developers

1. **State Preservation**: The message list `messages` acts as the agent's short-term memory. It grows turn-by-turn with both reasoning blocks and tool outputs.
2. **Parallel Tool Calling**: If Claude needs information about two cities (e.g. "Compare Austin and Seattle"), `response.content` may contain multiple `ToolUseBlock` objects in a single turn. Iterating over all blocks ensures all tools are executed.
3. **Matching IDs**: Each `tool_result` must match the corresponding `tool_use_id` from Claude's request.